# VizWiz Image Captioning — Phase 2
## EfficientNet-B3 Encoder + Transformer Decoder

## 1. Imports & Configuration

In [1]:
import os, json, random, pickle, math
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from torchvision import transforms
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

import nltk
from nltk.translate.bleu_score import corpus_bleu
nltk.download('punkt', quiet=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


## 2. Load Prepared Data

In [2]:
DATA_DIR   = "/kaggle/input/notebooks/tungbi811/vizwiz-data-preparation-group-10/data"
IMAGE_DIR = f"{DATA_DIR}/images"
 
# Load all pkl files
with open(f"{DATA_DIR}/word2idx.pkl", "rb") as f:
    word2idx = pickle.load(f)

with open(f"{DATA_DIR}/idx2word.pkl", "rb") as f:
    idx2word = pickle.load(f)
 
with open(f"{DATA_DIR}/train_tokenised.pkl", "rb") as f:
    train_tokenised = pickle.load(f)
 
with open(f"{DATA_DIR}/val_tokenised.pkl", "rb") as f:
    val_tokenised = pickle.load(f)
 
with open(f"{DATA_DIR}/test_tokenised.pkl", "rb") as f:
    test_tokenised = pickle.load(f)
 
with open(f"{DATA_DIR}/constants.pkl", "rb") as f:
    constants = pickle.load(f)
 
# Unpack constants for convenience
VOCAB_SIZE = constants['VOCAB_SIZE']
MAX_LEN    = constants['MAX_LEN']
PAD_IDX    = constants['PAD_IDX']
START_IDX  = constants['START_IDX']
END_IDX    = constants['END_IDX']
UNK_IDX    = constants['UNK_IDX']

# Image split directories
TRAIN_IMG_DIR = f"{IMAGE_DIR}/train"
VAL_IMG_DIR   = f"{IMAGE_DIR}/val"
TEST_IMG_DIR  = f"{IMAGE_DIR}/test"
 
# Verify everything loaded
print(f"Vocab size       : {VOCAB_SIZE}")
print(f"Train captions   : {len(train_tokenised)}")
print(f"Val captions     : {len(val_tokenised)}")
print(f"Test captions    : {len(test_tokenised)}")
print(f"Train images     : {len(os.listdir(TRAIN_IMG_DIR))}")
print(f"Val images       : {len(os.listdir(VAL_IMG_DIR))}")
print(f"Test images      : {len(os.listdir(TEST_IMG_DIR))}")
print()

Vocab size       : 3230
Train captions   : 23176
Val captions     : 4942
Test captions    : 5027
Train images     : 5279
Val images       : 1131
Test images      : 1132



## 3. Dataset & DataLoaders

In [3]:
# Custom Dataset class
class VizWizDataset(Dataset):
    def __init__(self, captions, image_dir, transform=None):
        self.captions  = captions
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.captions)

    def __getitem__(self, idx):
        item  = self.captions[idx]
        image = Image.open(os.path.join(self.image_dir, item['file_name'])).convert("RGB")
        if self.transform:
            image = self.transform(image)
        tokens = torch.tensor(item['tokens'], dtype=torch.long)
        return image, tokens


class CapsCollate:
    """Dynamically pads variable-length captions within each batch."""
    def __init__(self, pad_idx):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        imgs    = torch.cat([item[0].unsqueeze(0) for item in batch], dim=0)
        targets = pad_sequence([item[1] for item in batch],
                               batch_first=True, padding_value=self.pad_idx)
        return imgs, targets

In [4]:
# Calculate mean and std of VizWiz training images
plain_transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor()  # converts to [0,1]
])

plain_dataset = VizWizDataset(train_tokenised, TRAIN_IMG_DIR, transform=plain_transform)
plain_loader  = DataLoader(plain_dataset, batch_size=128, shuffle=False, num_workers=2)

mean = torch.zeros(3)
std  = torch.zeros(3)
n    = 0

for images, _ in tqdm(plain_loader):
    batch_size = images.size(0)
    images = images.view(batch_size, 3, -1)  # flatten H*W
    mean += images.mean(2).sum(0)
    std  += images.std(2).sum(0)
    n    += batch_size

mean /= n
std  /= n

print(f"Mean : {mean.tolist()}")
print(f"Std  : {std.tolist()}")

  0%|          | 0/363 [00:00<?, ?it/s]

Mean : [0.46644482016563416, 0.423868328332901, 0.3751975893974304]
Std  : [0.24309377372264862, 0.2424059510231018, 0.23670931160449982]


In [5]:
# Image transforms
train_transform = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.RandomCrop(300),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

val_test_transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [11]:
# Create datasets and dataloaders
BATCH_SIZE = 128
collate_fn = CapsCollate(pad_idx=word2idx["<PAD>"])

train_dataset = VizWizDataset(train_tokenised, TRAIN_IMG_DIR, transform=train_transform)
val_dataset   = VizWizDataset(val_tokenised,   VAL_IMG_DIR,   transform=val_test_transform)
test_dataset  = VizWizDataset(test_tokenised,  TEST_IMG_DIR,  transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True, collate_fn=collate_fn)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

Train batches : 182
Val batches   : 39
Test batches  : 40


In [12]:
img_root = "/kaggle/input/notebooks/tungbi811/vizwiz-data-preparation-group-10/data/images"
print("Contents of images folder:")
print(os.listdir(img_root))

# Check if subdirectories exist
for split in ['train', 'val', 'test']:
    path = os.path.join(img_root, split)
    if os.path.exists(path):
        print(f"{split}: {len(os.listdir(path))} files")
    else:
        print(f"{split}: NOT FOUND")

Contents of images folder:
['val', 'test', 'train']
train: 5279 files
val: 1131 files
test: 1132 files


In [13]:
images, tokens = next(iter(train_loader))
print(f"Image batch shape  : {images.shape}")
print(f"Tokens batch shape : {tokens.shape}")

Image batch shape  : torch.Size([128, 3, 300, 300])
Tokens batch shape : torch.Size([128, 25])


## 4. Model Architecture
### 4.1 EfficientNet-B3 Encoder

In [15]:
class EncoderEfficientNet(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        base = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
        self.features = base.features

        # Freeze all layers
        for param in self.features.parameters():
            param.requires_grad = False

        # Unfreeze last 2 blocks for fine-tuning
        for param in self.features[6].parameters():
            param.requires_grad = True
        for param in self.features[7].parameters():
            param.requires_grad = True

        # Project 1536 channels → embed_dim, output 100 spatial tokens
        self.projection = nn.Sequential(
            nn.AdaptiveAvgPool2d((10, 10)),
            nn.Flatten(2),
            nn.Conv1d(1536, embed_dim, 1)
        )

    def forward(self, x):
        features = self.features(x)          # (batch, 1536, H, W)
        out      = self.projection(features) # (batch, embed_dim, 100)
        return out.permute(0, 2, 1)          # (batch, 100, embed_dim)

In [16]:
# Test encoder
EMBED_DIM = 256
encoder   = EncoderEfficientNet(embed_dim=EMBED_DIM).to(DEVICE)

total     = sum(p.numel() for p in encoder.parameters())
trainable = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
print(f"Total params     : {total:,}")
print(f"Trainable params : {trainable:,}")
print(f"Frozen params    : {total - trainable:,}")

dummy = torch.randn(2, 3, 300, 300).to(DEVICE)
out   = encoder(dummy)
print(f"\nEncoder output shape: {out.shape}")  # expect (2, 100, 256)

Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 146MB/s] 


Total params     : 11,089,704
Trainable params : 8,306,654
Frozen params    : 2,783,050

Encoder output shape: torch.Size([2, 100, 256])


### 4.2 Positional Encoding

In [17]:
# Positional Encoding
class PositionalEncoding(nn.Module):
    """
    Injects position information into token embeddings.
    Uses fixed sinusoidal encoding (no learnable parameters).
    """
    def __init__(self, embed_dim, max_len=200, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe       = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() *
                             (-math.log(10000.0) / embed_dim))

        pe[:, 0::2] = torch.sin(position * div_term)  # even dims
        pe[:, 1::2] = torch.cos(position * div_term)  # odd dims
        pe = pe.unsqueeze(0)                           # (1, max_len, embed_dim)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

Positional encoding output shape: torch.Size([2, 25, 256])


In [ ]:
pe  = PositionalEncoding(embed_dim=EMBED_DIM).to(DEVICE)
dummy = torch.randn(2, 25, EMBED_DIM).to(DEVICE)
out   = pe(dummy)
print(f"Positional encoding output shape: {out.shape}")  # expect (2, 25, 256)

### 4.3 Transformer Decoder

In [19]:
NUM_HEADS  = 8
NUM_LAYERS = 3

decoder = DecoderTransformer(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS
).to(DEVICE)

dummy_captions  = torch.randint(0, VOCAB_SIZE, (2, 25)).to(DEVICE)
dummy_enc_out   = torch.randn(2, 100, EMBED_DIM).to(DEVICE)
out             = decoder(dummy_captions, dummy_enc_out)
print(f"Decoder output shape: {out.shape}")  # expect (2, 25, VOCAB_SIZE)

Decoder output shape: torch.Size([2, 25, 3230])


In [18]:
class DecoderTransformer(nn.Module):
    """
    Transformer decoder that attends to encoder image features
    and generates captions token by token.
    """
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.pos_encoding = PositionalEncoding(embed_dim, dropout=dropout)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(embed_dim, vocab_size)

    def forward(self, captions, encoder_out, tgt_mask=None, tgt_key_padding_mask=None):
        # captions: (batch, seq_len)
        x = self.embedding(captions)           # (batch, seq_len, embed_dim)
        x = self.pos_encoding(x)               # (batch, seq_len, embed_dim)
        out = self.transformer_decoder(
            x, encoder_out,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask
        )                                      # (batch, seq_len, embed_dim)
        return self.fc_out(out)                # (batch, seq_len, vocab_size)

### 4.4 Full Model

In [20]:
class ImageCaptioningModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, dropout=0.1):
        super().__init__()
        self.encoder = EncoderEfficientNet(embed_dim)
        self.decoder = DecoderTransformer(vocab_size, embed_dim, num_heads, num_layers, dropout)

    def forward(self, images, captions):
        encoder_out          = self.encoder(images)
        tgt_mask             = self.make_tgt_mask(captions.size(1)).to(images.device)
        tgt_key_padding_mask = (captions == PAD_IDX)
        return self.decoder(captions, encoder_out, tgt_mask, tgt_key_padding_mask)

    @staticmethod
    def make_tgt_mask(seq_len):
        """Causal mask — prevents decoder from attending to future tokens."""
        return torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()

In [21]:
model = ImageCaptioningModel(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS
).to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {total:,}")
print(f"Trainable params : {trainable:,}")

dummy_imgs     = torch.randn(2, 3, 300, 300).to(DEVICE)
dummy_captions = torch.randint(0, VOCAB_SIZE, (2, 25)).to(DEVICE)
out            = model(dummy_imgs, dummy_captions)
print(f"\nFull model output shape: {out.shape}")  # expect (2, 25, 3230)

Total params     : 15,907,014
Trainable params : 13,123,964

Full model output shape: torch.Size([2, 25, 3230])


## 5. Training
### 5.1 Loss Function & Optimizer

In [22]:
LEARNING_RATE = 3e-4
NUM_EPOCHS    = 10

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

print(f"Loss     : CrossEntropyLoss (ignore PAD)")
print(f"Optimizer: Adam  (lr={LEARNING_RATE})")
print(f"Scheduler: ReduceLROnPlateau (patience=2)")
print(f"Epochs   : {NUM_EPOCHS}")

Loss     : CrossEntropyLoss (ignore PAD)
Optimizer: Adam  (lr=0.0003)
Scheduler: ReduceLROnPlateau (patience=2)
Epochs   : 10


### 5.2 Training & Validation Loop

In [23]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for images, captions in loader:
        images   = images.to(DEVICE)
        captions = captions.to(DEVICE)

        # Input: all tokens except last, Target: all tokens except first
        inputs  = captions[:, :-1]
        targets = captions[:, 1:]

        optimizer.zero_grad()
        outputs = model(images, inputs)  # (batch, seq_len-1, vocab_size)

        # Reshape for CrossEntropyLoss
        loss = criterion(
            outputs.reshape(-1, VOCAB_SIZE),
            targets.reshape(-1)
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)

In [24]:
def validate(model, loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, captions in loader:
            images   = images.to(DEVICE)
            captions = captions.to(DEVICE)

            inputs  = captions[:, :-1]
            targets = captions[:, 1:]

            outputs = model(images, inputs)
            loss    = criterion(
                outputs.reshape(-1, VOCAB_SIZE),
                targets.reshape(-1)
            )
            total_loss += loss.item()

    return total_loss / len(loader)

### 5.3 Train

In [ ]:
train_losses = []
val_losses   = []
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss   = validate(model, val_loader, criterion)
    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pth")

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]  "
          f"Train Loss: {train_loss:.4f}  "
          f"Val Loss: {val_loss:.4f}  "
          f"LR: {optimizer.param_groups[0]['lr']:.6f}")

print(f"\nBest Val Loss: {best_val_loss:.4f}")